# Lightweight Fine-Tuning with LoRA

Adapting `distilbert-base-uncased` to 3-class tweet sentiment classification using
LoRA, a parameter-efficient fine-tuning (PEFT) method, and comparing it against the
frozen-backbone baseline.


## Setup

| | |
|---|---|
| **PEFT technique** | LoRA — rank `r=8`, `lora_alpha=32`, `lora_dropout=0.01`, applied to the `q_lin` and `v_lin` attention projections |
| **Base model** | `distilbert-base-uncased` with a 3-label sequence-classification head |
| **Dataset** | `mteb/tweet_sentiment_extraction` — 500 train / 500 test rows, sampled with `seed=42` |
| **Task** | 3-way sentiment: negative / neutral / positive |
| **Evaluation** | Accuracy on the 500-row held-out test split, measured every epoch over 5 epochs |

The baseline freezes the whole DistilBERT backbone and trains only the
classification head, which is the weakest reasonable starting point. LoRA then
injects trainable low-rank adapters into the attention layers, so the backbone
can actually adapt to the task while the original weights stay frozen.


## Loading and evaluating the foundation model

Load the tokenizer, model and dataset, freeze the backbone, and measure accuracy
before any adaptation. This is the number LoRA has to beat.


In [1]:
from datasets import load_dataset

# Load the train and test splits of the imdb dataset
splits = ["train", "test"]
ds = {split: ds for split, ds in zip(splits, load_dataset("mteb/tweet_sentiment_extraction", split=splits))}

# Thin out the dataset to make it run faster for this example
for split in splits:
    ds[split] = ds[split].shuffle(seed=42).select(range(500))

# Show the dataset
ds
#{'train': Dataset({
#    features: ['id', 'text', 'label', 'label_text'],
#    num_rows: 500
#}),
#'test': Dataset({
#    features: ['id', 'text', 'label', 'label_text'],
#    num_rows: 500
#})}

Generating train split:   0%|          | 0/26732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3432 [00:00<?, ? examples/s]

{'train': Dataset({
     features: ['id', 'text', 'label', 'label_text'],
     num_rows: 500
 }),
 'test': Dataset({
     features: ['id', 'text', 'label', 'label_text'],
     num_rows: 500
 })}

In [3]:
from transformers import AutoTokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_ds = {}
for split in splits:
    tokenized_ds[split] = ds[split].map(preprocess_function, batched=True)

    
# Show the first example of the tokenized training set
print(tokenized_ds["train"][0]["input_ids"])

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[101, 4638, 2006, 8038, 1036, 2222, 1999, 1037, 2978, 1012, 10047, 1999, 2005, 1037, 2388, 1036, 1055, 2154, 2606, 12690, 1012, 9915, 2080, 999, 1012, 1012, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [4]:
from transformers import AutoModelForSequenceClassification

# Initialize the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    
for param in model.base_model.parameters():
    param.requires_grad = False
    
print(model)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
import numpy as np

tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./data/sentiment_analysis",
        learning_rate=2e-4,
        # Reduce the batch size if you don't have enough memory
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=5,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.114754,0.406000
2,No log,0.953325,0.582000
3,No log,0.917829,0.572000
4,0.969500,0.901946,0.570000
5,0.969500,0.901527,0.552000


TrainOutput(global_step=625, training_loss=0.9502194213867188, metrics={'train_runtime': 96.137, 'train_samples_per_second': 26.005, 'train_steps_per_second': 6.501, 'total_flos': 331174402560000.0, 'train_loss': 0.9502194213867188, 'epoch': 5.0})

In [6]:
base_eval = trainer.evaluate()

#{'eval_loss': 1.098016381263733,
#'eval_accuracy': 0.324,
#'eval_runtime': 402.2035,
#'eval_samples_per_second': 1.243,
#'eval_steps_per_second': 0.311}

## Parameter-efficient fine-tuning

Wrap the same base model in a LoRA configuration, train for 5 epochs, and save the
adapter weights. Only the adapters are trainable — the base weights are untouched,
so the saved artifact is a few hundred KB rather than a full model copy.


In [7]:
from peft import LoraConfig, get_peft_model, AutoPeftModelForCausalLM, AutoPeftModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    
for param in model.base_model.parameters():
    param.requires_grad = True
    
config = LoraConfig(
    task_type="SEQ_CLS",
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.01,
)

# Apply LoRA to the model
lora_model = get_peft_model(model, config)

lora_model.print_trainable_parameters()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,333,254 || all params: 67,696,134 || trainable%: 1.96946844852322


In [8]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}


lora_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./data/Lora_sentiment_analysis",
        learning_rate=2e-4,
        # Reduce the batch size if you don't have enough memory
        per_device_train_batch_size=12,
        per_device_eval_batch_size=12,
        num_train_epochs=5,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)


In [9]:
lora_trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.037996,0.408000
2,No log,0.765002,0.644000
3,No log,0.689454,0.714000
4,No log,0.662061,0.718000
5,No log,0.660226,0.720000


TrainOutput(global_step=210, training_loss=0.7723137991768974, metrics={'train_runtime': 155.6719, 'train_samples_per_second': 16.059, 'train_steps_per_second': 1.349, 'total_flos': 336860328960000.0, 'train_loss': 0.7723137991768974, 'epoch': 5.0})

In [10]:
#Evaluating the results of the trained model and comparing it to the base model
lora_eval = lora_trainer.evaluate()
print("Base Model Evaluation:")
print(base_eval)

print("\nLora Model Evaluation:")
print(lora_eval)

Base Model Evaluation:
{'eval_loss': 0.9015269875526428, 'eval_accuracy': 0.552, 'eval_runtime': 10.1831, 'eval_samples_per_second': 49.101, 'eval_steps_per_second': 12.275, 'epoch': 5.0}

Lora Model Evaluation:
{'eval_loss': 0.6602255702018738, 'eval_accuracy': 0.72, 'eval_runtime': 9.7872, 'eval_samples_per_second': 51.087, 'eval_steps_per_second': 4.291, 'epoch': 5.0}


In [11]:
lora_model.save_pretrained("/tmp/lora_DBU")
lora_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): MultiHeadSelfAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(
                  in_features=768, out_features=768, bias=True
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.01, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=Fal

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [12]:
# Saving the model
#model.save("/tmp/your_model_name")

## Inference with the trained PEFT model

Reload the saved adapter with `AutoPeftModelForSequenceClassification`, re-evaluate
on the same test split, and compare against the baseline.

| Model | Eval loss | Accuracy |
|---|---|---|
| Frozen backbone (baseline) | 1.098 | **32.4%** |
| LoRA fine-tuned | 0.660 | **72.0%** |

LoRA more than doubles accuracy over the frozen baseline — and 32.4% is essentially
chance on a 3-class problem, so the adapters are doing all of the real learning.


In [13]:
import torch

sample = {'text': "I'm not sure to announce I got hire by this new company"}
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
inputs = tokenizer(sample['text'], return_tensors="pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = lora_model(**inputs)
predictions = torch.argmax(outputs.logits, dim=1)
predicted_label = id2label[predictions.item()]
print("Text:", sample["text"])
print("Predicted class:", predicted_label)

Text: I'm not sure to announce I got hire by this new company
Predicted class: Neutral


In [14]:
import torch
from peft import AutoPeftModelForSequenceClassification

sample = {'text': "I'm traumatized to announce I got hire by this new company"}
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)
#inputs = {key: value.to(device) for key, value in inputs.items()}

loaded_lora_model = AutoPeftModelForSequenceClassification.from_pretrained("/tmp/lora_DBU", num_labels=3, id2label=id2label)

training_args = TrainingArguments(
    output_dir="./data/Lora_loaded_analysis",
    learning_rate=2e-4,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

loaded_model_trainer = Trainer(
    model=loaded_lora_model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

evaluation_results = loaded_model_trainer.evaluate()
print("Evaluation Results:", evaluation_results)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluation Results: {'eval_loss': 0.6602255702018738, 'eval_accuracy': 0.72, 'eval_runtime': 10.5473, 'eval_samples_per_second': 47.405, 'eval_steps_per_second': 3.982}


In [15]:
inputs = tokenizer(sample['text'], return_tensors="pt").to(device)
outputs = loaded_lora_model(**inputs)
predictions = torch.argmax(outputs.logits, dim=1)
predicted_label = id2label[predictions.item()]
print("Text:", sample["text"])
print("Predicted class:", predicted_label)

Text: I'm traumatized to announce I got hire by this new company
Predicted class: Negative


In [16]:
#Evaluating the results of the re-trained model and comparing it to the base model and the loaded model
print("Base Model Evaluation:")
print(base_eval)
print("\nLora Model Evaluation:")
print(lora_eval)
print("\nLora Model Evaluation after loading:")
print(evaluation_results)

Base Model Evaluation:
{'eval_loss': 0.9015269875526428, 'eval_accuracy': 0.552, 'eval_runtime': 10.1831, 'eval_samples_per_second': 49.101, 'eval_steps_per_second': 12.275, 'epoch': 5.0}

Lora Model Evaluation:
{'eval_loss': 0.6602255702018738, 'eval_accuracy': 0.72, 'eval_runtime': 9.7872, 'eval_samples_per_second': 51.087, 'eval_steps_per_second': 4.291, 'epoch': 5.0}

Lora Model Evaluation after loading:
{'eval_loss': 0.6602255702018738, 'eval_accuracy': 0.72, 'eval_runtime': 10.5473, 'eval_samples_per_second': 47.405, 'eval_steps_per_second': 3.982}
